In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path
import fiona
import datetime

In [2]:
input_dir = Path.cwd() / "input_aanpassingen_db"
output_dir = Path.cwd() / "output_aanpassingen_db"

In [3]:
huidig = input_dir / "kansendatabase_huidige.gpkg"
nieuw = input_dir / "kansendatabase_nieuw.gpkg"

In [4]:
# Check dat de lagen goed worden ingelezen
layers = fiona.listlayers(huidig)
assert set(layers) == set(
    [
        "primaire_keringen",
        "niet_primaire_keringen",
        "doorbraak_locaties_primair",
        "doorbraak_locaties_regionaal",
    ]
)

In [5]:
gdf_primair_kansen_huidig = gpd.read_file(huidig, layer="primaire_keringen")
gdf_niet_primair_huidig = gpd.read_file(huidig, layer="niet_primaire_keringen")
gdf_doorbraak_locaties_primair_huidig = gpd.read_file(
    huidig,
    layer="doorbraak_locaties_primair",
)
gdf_doorbraak_locaties_regionaal_huidig = gpd.read_file(
    huidig,
    layer="doorbraak_locaties_regionaal",
)

In [6]:
gdf_primair_kansen_nieuw = gpd.read_file(nieuw, layer="primaire_keringen")
gdf_niet_primair_nieuw = gpd.read_file(nieuw, layer="niet_primaire_keringen")
gdf_doorbraak_locaties_primair_nieuw = gpd.read_file(
    nieuw,
    layer="doorbraak_locaties_primair",
)
gdf_doorbraak_locaties_regionaal_nieuw = gpd.read_file(
    nieuw,
    layer="doorbraak_locaties_regionaal",
)

In [7]:
logbook = "# Loggboek verschillen kansendatabase\n\n"
logbook += f'huidige versie: {huidig.name} vs  nieuwe verseie: {nieuw.name}\n\n'
logbook += f"### {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n"

Deze vergelijking is op basis van de kollom fid in qgis, dus geen aanpassingen maken aan die kolom

Geopandas leest deze standaard in

In [8]:
# sanity checks on columns and dtypes for all 4 dataset pairs
datasets = [
    ("primair_kansen", gdf_primair_kansen_huidig, gdf_primair_kansen_nieuw),
    ("niet_primair", gdf_niet_primair_huidig, gdf_niet_primair_nieuw),
    (
        "doorbraak_locaties_primair",
        gdf_doorbraak_locaties_primair_huidig,
        gdf_doorbraak_locaties_primair_nieuw,
    ),
    (
        "doorbraak_locaties_regionaal",
        gdf_doorbraak_locaties_regionaal_huidig,
        gdf_doorbraak_locaties_regionaal_nieuw,
    ),
]

logbook += "\nCheck columns and dtypes\n"
for name, gdf_huidig, gdf_nieuw in datasets:
    if set(gdf_huidig.columns) != set(gdf_nieuw.columns):
        logbook += f"[{name}] Columns are different\n"  
        verschil_huidig = set(gdf_huidig.columns) - set(gdf_nieuw.columns)
        verschil_nieuw = set(gdf_nieuw.columns) - set(gdf_huidig.columns)

        if len(verschil_huidig) > 0:
                logbook += f"alleen huidig: {verschil_huidig}\n"
        if len(verschil_nieuw) > 0:
                logbook += f"alleen nieuw: {verschil_nieuw}\n"
    
        # Alleen gedeelde kolommen vergelijken op dtype
        for col_name in (set(gdf_huidig.columns) & set(gdf_nieuw.columns)):
            if gdf_huidig[col_name].dtype != gdf_nieuw[col_name].dtype:
                logbook += (
                    f"[{name}] Column {col_name} has different dtypes:\n"
                    f"{gdf_huidig[col_name].dtype} vs {gdf_nieuw[col_name].dtype}\n"
                )

In [9]:
# check nieuwe indices, handel
logbook+= "\n ### Check new rows\n"
for name, gdf_huidig, gdf_nieuw in datasets:
    if set(gdf_huidig.index) != set(gdf_nieuw.index):
        logbook += f"[{name}] Verschillende indices:\n"
        verschil_huidig = set(gdf_huidig.index) - set(gdf_nieuw.index)
        verschil_nieuw = set(gdf_nieuw.index) - set(gdf_huidig.index)
        if len(verschil_huidig) > 0:
            logbook += f"verwijderde rijen: {verschil_huidig}\n"
            verwijderde_rijen = gdf_huidig.loc[list(verschil_huidig)]
            for idx, row in verwijderde_rijen.iterrows():
                logbook += f"{row.T}\n"
        if len(verschil_nieuw) > 0:
            logbook += f"Nieuwe rijen: {verschil_nieuw}\n"
            nieuwe_rijen = gdf_nieuw.loc[list(verschil_nieuw)]
            for idx, row in nieuwe_rijen.iterrows():
                logbook += f"{row.T}\n"

In [10]:
# check nieuwe indices, handel
logbook+= "\n### Check row differences\n"
lst_joined = []
for name, gdf_huidig, gdf_nieuw in datasets:
    # eerst alleen de inhoud
    joined_df: pd.DataFrame = gdf_huidig.join(gdf_nieuw, lsuffix='_huidig', rsuffix='_nieuw')
    common_cols = gdf_huidig.columns.intersection(gdf_nieuw.columns)
    diff_mask = {}

    for c in common_cols:
        left = joined_df[f"{c}_huidig"]
        right = joined_df[f"{c}_nieuw"]

        if c == 'geometry':  # geometry
            equal = left.geom_equals(right)
        else:
            equal = left.eq(right) | (left.isna() & right.isna())

        diff_mask[c] = ~equal

    diff_mask = pd.DataFrame(diff_mask, index=joined_df.index)
    changed_rows = diff_mask.any(axis=1)
    changed_cols = diff_mask.columns[diff_mask.any(axis=0)].tolist()

    if changed_rows.any():
        logbook += f"[{name}] {int(changed_rows.sum())} gewijzigde rijen; kolommen: {changed_cols}\n"
        cols_to_keep = [f"{c}_huidig" for c in changed_cols] + [f"{c}_nieuw" for c in changed_cols]
        df_differences = joined_df.loc[changed_rows, cols_to_keep]
        lst_joined.append((name, df_differences))
        for idx, row in df_differences.iterrows():
                logbook += f"{row.T}\n"
    else:
        lst_joined.append((name, joined_df.iloc[0:0]))

In [11]:
from IPython.display import Markdown, display

display(Markdown(logbook))

# Loggboek verschillen kansendatabase

huidige versie: kansendatabase_huidige.gpkg vs  nieuwe verseie: kansendatabase_nieuw.gpkg

### 2026-03-13 14:21


Check columns and dtypes
[primair_kansen] Columns are different
alleen nieuw: {'test col'}

 ### Check new rows
[primair_kansen] Verschillende indices:
Nieuwe rijen: {721}
TRAJECT_ID                                                #35-31
TRAJECT_DL                                                036013
DUIDING          Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar
klasse                                                       4.0
KANS                                                       562.0
NORMOG_2050                                                 3000
test col                                                     NaN
geometry       MULTILINESTRING ((134268.03521876037 412893.10...
Name: 721, dtype: object

### Check row differences
[primair_kansen] 1 gewijzigde rijen; kolommen: ['DUIDING', 'KANS', 'NORMOG_2050', 'geometry']
DUIDING_huidig                  Kleine kans: 1/300 tot 1/3.000 per jaar
KANS_huidig                                                       719.0
NORMOG_2050_huidig                                                  300
geometry_huidig       MULTILINESTRING ((109372.49439999834 414781.54...
DUIDING_nieuw                 Kleine kans: 1/3000 tot 1/30.000 per jaar
KANS_nieuw                                                       3210.0
NORMOG_2050_nieuw                                                  3000
geometry_nieuw        MULTILINESTRING ((109372.49439999834 414781.54...
Name: 415, dtype: object
